# Lot 44 — Validation hors échantillon du routeur MCTS 32768 → 65536

Notebook **mince** : toute la logique est dans `apps/trainer/scripts/lot44/` et le runner
`run_lot44_router_oos_validation.py`. Aucune modification de code n'est nécessaire entre les étapes :
seule la cellule **CONFIGURATION** se modifie.

| Étape | Effet | Bloquant |
|---|---|---|
| 1 Mount Drive | monte Google Drive, vérifie `songo-ai/` | oui |
| 2 Locate project | clone/MAJ du dépôt (branche configurée) + extraction vérifiée de `lot44_inputs.tar.gz` | oui |
| 3 Environment | installation, versions, CUDA | oui |
| 4 Preflight | `selftest` (pytest Lot44) puis `preflight` réel → **STOP si FAIL** | oui |
| 5 Smoke test | pipeline complet miniature (budgets 4/8/16/256) | oui |
| 6 Prepare Lot44 | corpus OOS, split par partie, `test_lock.json`, plan + analyse de puissance | oui |
| 7 Execute / Resume | recherches → gel du routeur → 65536 TEST → évaluation unique (relancer = reprise) | `CONFIRM_LONG_COMPUTE` |
| 8 Finalize | vérifications, comptabilité RESTART/RESUME, `decision.json` | oui |
| 9 Export | `lot44_results.tar.gz` vérifié dans `songo-ai/exports/` | oui |

Aucun entraînement SRN/G5, aucune modification moteur/MCTS, aucun label Minimax.

In [ ]:
# ===================== CONFIGURATION (seule cellule à modifier) =====================
REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_BRANCH = 'dev'            # le code Lot44 vit sur dev
EXPECTED_COMMIT = ''           # optionnel : SHA exact à exiger ('' = pas de contrôle)
DRIVE_ROOT_OVERRIDE = ''       # '' = /content/drive/MyDrive
PROJECT_ROOT_OVERRIDE = ''     # '' = /content/songo-fish
DEVICE = 'auto'                # 'auto' | 'cuda' | 'cpu'
SEED = 20264401
MAX_POSITIONS = 768            # plafond du corpus OOS (voir l'analyse de puissance à l'étape 6)
CONCURRENCY = None             # None = défauts par budget (4096/8192: 256, 32768: 128, 65536: 64)
MAX_SHARDS_PER_SESSION = None  # None = tout ; un entier = découpe la session (reprise ensuite)
CONFIRM_LONG_COMPUTE = False   # mettre True pour lancer l'étape 7 (calcul long)
# ====================================================================================

## STEP 1 — Mount Drive

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, hashlib, tarfile

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path(DRIVE_ROOT_OVERRIDE or '/content/drive/MyDrive')
SONGO_AI = DRIVE_ROOT / 'songo-ai'
if not SONGO_AI.is_dir():
    raise RuntimeError(f'songo-ai introuvable sous {DRIVE_ROOT} : vérifier DRIVE_ROOT_OVERRIDE')
INPUT_BUNDLE = SONGO_AI / 'inputs' / 'lot44_inputs.tar.gz'
print('DRIVE_ROOT =', DRIVE_ROOT)
print('INPUT_BUNDLE =', INPUT_BUNDLE, 'OK' if INPUT_BUNDLE.is_file() else 'MISSING')

## STEP 2 — Locate project
Clone (ou met à jour) la branche configurée, puis extrait les inputs Lot44 après vérification SHA-256 (bundle **et** chaque fichier).

In [ ]:
REPO_DIR = Path(PROJECT_ROOT_OVERRIDE or '/content/songo-fish')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'merge', '--ff-only', f'origin/{REPO_BRANCH}'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if EXPECTED_COMMIT and not COMMIT.startswith(EXPECTED_COMMIT):
    raise RuntimeError(f'commit {COMMIT} != EXPECTED_COMMIT {EXPECTED_COMMIT}')
RUNNER = REPO_DIR / 'apps/trainer/scripts/run_lot44_router_oos_validation.py'
if not RUNNER.is_file() or not (REPO_DIR / 'apps/trainer/scripts/lot44/pipeline.py').is_file():
    raise RuntimeError(f'pipeline Lot44 v2 (apps/trainer/scripts/lot44/) absent sur origin/{REPO_BRANCH} @ {COMMIT[:10]} : pousser le code Lot44 sur {REPO_BRANCH}')

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not INPUT_BUNDLE.is_file():
    raise RuntimeError(f'{INPUT_BUNDLE} absent : lancer en local `--stage prepare-inputs --sync-to-drive`')
expected = Path(str(INPUT_BUNDLE) + '.sha256').read_text().split()[0]
if file_sha256(INPUT_BUNDLE) != expected:
    raise RuntimeError('checksum du bundle inputs Lot44 invalide')
with tarfile.open(INPUT_BUNDLE, 'r:gz') as archive:
    archive.extractall(REPO_DIR)
manifest = json.loads((REPO_DIR / 'lot44_input_manifest.json').read_text())
bad = [f for f, digest in manifest['files'].items() if file_sha256(REPO_DIR / f) != digest]
if bad:
    raise RuntimeError(f'fichiers inputs corrompus après extraction : {bad}')
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR, '| branch =', REPO_BRANCH, '| commit =', COMMIT)
print('inputs vérifiés :', len(manifest['files']), 'fichiers')

## STEP 3 — Environment

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev]'], check=True)
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA_AVAILABLE =', 'YES' if torch.cuda.is_available() else 'NO')
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
elif DEVICE == 'cuda':
    raise RuntimeError("DEVICE='cuda' mais CUDA indisponible : changer de runtime ou DEVICE")

BASE = ['--drive-root', str(DRIVE_ROOT), '--device', DEVICE, '--seed', str(SEED), '--max-positions', str(MAX_POSITIONS)]
COMPUTE = (['--concurrency', str(CONCURRENCY)] if CONCURRENCY else []) + (['--max-shards', str(MAX_SHARDS_PER_SESSION)] if MAX_SHARDS_PER_SESSION else [])
OUTPUT = SONGO_AI / 'experiments' / 'lot44_router_out_of_sample_validation'

def lot44(stage, *extra):
    """Lance un stage du runner en diffusant sa sortie ; lève une erreur si le stage échoue."""
    cmd = [sys.executable, '-u', str(RUNNER), '--stage', stage, *BASE, *extra]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'Lot44 stage {stage} FAILED (code {proc.returncode}); voir {OUTPUT}/errors.jsonl')

print(json.dumps({'branch': REPO_BRANCH, 'commit': COMMIT, 'device': DEVICE, 'seed': SEED, 'max_positions': MAX_POSITIONS, 'concurrency': CONCURRENCY or 'defaults', 'max_shards_per_session': MAX_SHARDS_PER_SESSION, 'confirm_long_compute': CONFIRM_LONG_COMPUTE, 'output': str(OUTPUT)}, indent=2))

## STEP 4 — Preflight
`selftest` exécute la suite pytest Lot44 (`test_report.json`), puis `preflight` exécute toutes les vérifications réelles. **Le notebook s'arrête ici si PREFLIGHT_STATUS ≠ PASS.**

In [ ]:
lot44('selftest')
lot44('preflight')
PREFLIGHT = json.loads((OUTPUT / 'preflight_report.json').read_text())
for name, status in PREFLIGHT['summary'].items():
    print(f'{name:<24} {status}')
print('PREFLIGHT_STATUS =', PREFLIGHT['PREFLIGHT_STATUS'], '| SCIENTIFIC_RUN_ALLOWED =', PREFLIGHT['SCIENTIFIC_RUN_ALLOWED'])
if PREFLIGHT['PREFLIGHT_STATUS'] != 'PASS':
    raise RuntimeError('PREFLIGHT FAIL — STOP. Voir preflight_report.json et errors.jsonl')

## STEP 5 — Smoke test

In [ ]:
lot44('smoke')
SMOKE = json.loads((OUTPUT / 'smoke_test_report.json').read_text())
print('SMOKE_TEST =', SMOKE['SMOKE_TEST'], '| secondes =', SMOKE['seconds'])
if SMOKE['SMOKE_TEST'] != 'PASS':
    raise RuntimeError('SMOKE FAIL — STOP')

## STEP 6 — Prepare Lot44
Corpus OOS (0 overlap exigé), split par partie, `test_lock.json`, label/protocole gelés. Affiche la charge de calcul **et l'analyse de puissance** avant tout calcul long.

In [ ]:
lot44('prepare')
lot44('plan')
POWER = json.loads((OUTPUT / 'power_analysis.json').read_text())
OVERLAP = json.loads((OUTPUT / 'overlap_audit.json').read_text())
print('OOS_OVERLAP_WITH_ORIGINAL_256 =', OVERLAP['OOS_OVERLAP_WITH_ORIGINAL_256'], '| game overlap =', OVERLAP['OOS_GAME_OVERLAP_WITH_ORIGINAL_256'])
for name, s in POWER['scenarios'].items():
    print(f"{name}: positifs TEST attendus = {s['expected_test_positives']:.1f}, P(>= {POWER['min_oos_positives_for_verdict']}) = {s['p_test_positives_at_least_minimum']:.3f}")

## STEP 7 — Execute / Resume
Relancer cette cellule reprend exactement où le calcul s'est arrêté (shards terminés jamais recalculés). Le 65536 du TEST n'est lancé qu'après le gel du routeur ; l'évaluation TEST est unique.

In [ ]:
if not CONFIRM_LONG_COMPUTE:
    lot44('run')  # affiche le plan sans calculer
    raise RuntimeError('CONFIRM_LONG_COMPUTE = False : relire le plan ci-dessus puis passer True dans la CONFIGURATION')
lot44('run', '--confirm', *COMPUTE)
lot44('status')

## STEP 8 — Finalize

In [ ]:
lot44('finalize')
DECISION = json.loads((OUTPUT / 'decision.json').read_text())
print(json.dumps(DECISION, indent=2))

## STEP 9 — Export

In [ ]:
EXPORT = SONGO_AI / 'exports' / 'lot44_results.tar.gz'
lot44('export', '--bundle', str(EXPORT))
print('EXPORT =', EXPORT, '| sha256 =', Path(str(EXPORT) + '.sha256').read_text().split()[0])